# 04_load_fixed_and_public.py

```text
WHAT : Loads the "fixed broadband" and "public / last-resort access" layers.
       (1) NBN fixed-line and fixed-wireless footprint polygons, clipped to the NT.
       (2) RICT (community payphones / Wi-Fi phones) and STAND (disaster resilience sites)
           points in the NT, as one tidy public-access-point table.
READS: data/external/nbn/fixedline/nbn_coverage_fixedline.shp
       data/external/nbn/wireless/nbn_coverage_wireless.shp
       data/raw/rict/Com_RICT_Community_NIAA_2024.json
       data/external/stand/strengthening_telecommunication_against_natural_disaster.json
WRITES: data/processed/04_nbn_footprint.geojson       (2 polygons: technology = fixed_line / fixed_wireless)
        data/processed/04_public_access.csv           (+ .geojson, + .columns.md)
WHY  : A community is "fixed-broadband served" if its point sits inside an NBN fixed-line or
       fixed-wireless polygon (otherwise home internet is Sky Muster satellite or a non-NBN
       provider we cannot see). "Public access" = a RICT payphone/Wi-Fi phone or a STAND
       site in the community - the last-resort tier when there is no mobile and no fixed line.
RUN  : python src/04_load_fixed_and_public.py        (about 10 s)

The NBN shapefiles are in GDA94 (EPSG:4283); we reproject them to plain lat/long. They are
national, so we read only the NT bounding box.
```

In [7]:
import importlib
import json
import geopandas as gpd
import pandas as pd

In [22]:
STAND_JSON = "data/external/stand/strengthening_telecommunication_against_natural_disaster.json"
NBN_FIXEDLINE_SHP = 'data/external/nbn/fixedline/nbn_coverage_fixedline.shp'
NBN_WIRELESS_SHP = 'data/external/nbn/wireless/nbn_coverage_wireless.shp'
RICT_JSON = "data/raw/rict/Com_RICT_Community_NIAA_2024.json"


In [8]:
def _nt_points_from_geojson(path, want_state, state_key, name_key, detail_key, source_label):
    """Read a GeoJSON FeatureCollection of points, keep the NT ones, return a tidy DataFrame."""
    with open(path, encoding="utf-8") as fh:
        collection = json.load(fh)
    rows = []
    for feature in collection["features"]:
        geometry = feature.get("geometry") or {}
        if geometry.get("type") != "Point":
            continue
        prop = feature.get("properties") or {}
        if str(prop.get(state_key, "")).upper() != want_state:
            continue
        lon, lat = geometry["coordinates"][:2]
        rows.append({
            "source": source_label,
            "name": str(prop.get(name_key) or "").strip(),
            "latitude": lat,
            "longitude": lon,
            "detail": str(prop.get(detail_key) or "").strip(),
        })
    return pd.DataFrame(rows)

### 1. NBN fixed-line and fixed-wireless footprints, NT only.

In [10]:
nbn_gdf = gpd.read_file('data/external/nbn/fixedline/nbn_coverage_fixedline.shp')

In [11]:
nbn_gdf

,polygon_id,geometry
0,3MOO-22,"POLYGON ((144.42028 -38.1786, 144.42024 -38.17..."
1,4SRW-63,"POLYGON ((152.98842 -27.51775, 152.98744 -27.5..."
2,3NAW-62,"POLYGON ((145.31401 -38.02208, 145.3137 -38.02..."
3,2LID-03,"POLYGON ((151.03546 -33.8687, 151.03472 -33.86..."
4,2MDW-21,"POLYGON ((151.89023 -32.74489, 151.88942 -32.7..."
...,...,...
5311,5SMF-06,"POLYGON ((138.68333 -34.6995, 138.68246 -34.70..."
5312,5EZB-01,"POLYGON ((138.67336 -34.71546, 138.67287 -34.7..."
5313,2ROU-04,"POLYGON ((150.92873 -33.65985, 150.92857 -33.6..."
5314,4RVH-20,"POLYGON ((152.92181 -25.42655, 152.92178 -25.4..."


In [12]:
NT_BBOX = (128.9, -26.3, 138.1, -10.8) #Rough NT bounding box (lon_min, lat_min, lon_max, lat_max) - used to pre-filter 
# The ABS boundary shapefiles are published in GDA2020; the NBN shapefiles in GDA94.
CRS_NBN            = "EPSG:4283"

# Lat/long in degrees. All the point files (BushTel, RICT, KML, GeoJSON) are already in this, and we store our processed point files in it too.
CRS_LATLON = "EPSG:4326"

In [13]:
def _load_nbn(path, technology):
    gdf = gpd.read_file(path, bbox=NT_BBOX)      # pyogrio reads only the NT window
    if gdf.crs is None:
        gdf = gdf.set_crs(CRS_NBN)               # the .prj says GDA94
    gdf = gdf.to_crs(CRS_LATLON)[["geometry"]]
    gdf["technology"] = technology
    # merge all the little polygons for this technology into one shape
    return gdf.dissolve(by="technology").reset_index()

In [15]:
nbn = pd.concat([
    _load_nbn(NBN_FIXEDLINE_SHP, "fixed_line"),
    _load_nbn(NBN_WIRELESS_SHP, "fixed_wireless"),
], ignore_index=True)
nbn = gpd.GeoDataFrame(nbn, geometry="geometry", crs=CRS_LATLON)

In [16]:
nbn

,technology,geometry
0,fixed_line,"MULTIPOLYGON (((130.8413 -12.46742, 130.8411 -..."
1,fixed_wireless,"MULTIPOLYGON (((130.89923 -12.89625, 130.8997 ..."


In [17]:

nbn.to_file( "data/new_processed/nbn_footprint.geojson", driver="GeoJSON")
print(f"NBN footprint: {', '.join(nbn['technology'])} (dissolved to {len(nbn)} polygons)")

NBN footprint: fixed_line, fixed_wireless (dissolved to 2 polygons)


### 2. Public / last-resort access points: RICT + STAND, NT only.

In [ ]:
rict = _nt_points_from_geojson(
    RICT_JSON, want_state="NT",
    state_key="state", name_key="community_name", detail_key="site_type", source_label="rict",
)

In [19]:
rict

,source,name,latitude,longitude,detail
0,rict,20 Mile,-16.08575,136.54839,WP
1,rict,Adjamarragu,-11.15277,132.47481,WP
2,rict,Akwalirrumanja,-13.86360,136.77864,WP
3,rict,Alamirra,-11.06768,132.52817,WP
4,rict,Alice Well,-24.88131,134.22964,WP
...,...,...,...,...,...
291,rict,Mungkarta - WIFI,-20.33712,134.19820,WH
292,rict,Nyirripi - WIFI,-22.64972,130.55220,WH
293,rict,Tara - WIFI,-21.50685,133.99593,WH
294,rict,Wutunugurra - WIFI,-20.46263,135.25060,WH


In [ ]:
stand = _nt_points_from_geojson(
    STAND_JSON, want_state="NT",
    state_key="Jurisdiction", name_key="Site_Name", detail_key="Public_Access", source_label="stand",
)

In [21]:
stand

,source,name,latitude,longitude,detail
0,stand,Adelaide River School,-13.237010,131.108845,Yes
1,stand,Alyangula Area School,-13.849726,136.420121,Yes
2,stand,Alyarrmandumanja Umbakumba School,-13.859394,136.812703,Yes
3,stand,Amanbidji School,-16.427548,129.618066,Yes
4,stand,Angurugu School,-13.982530,136.459330,Yes
...,...,...,...,...,...
83,stand,Alyangula Health Clinic,-13.848046,136.421216,Yes
84,stand,Alyangula Police Station,-13.852252,136.421650,Yes
85,stand,Port Keats (Wadeye) Health Clinic,-14.241205,129.520241,Yes
86,stand,Port Keats (Wadeye) Police Station,-14.239996,129.520903,Yes


In [23]:
public_access = pd.concat([rict, stand], ignore_index=True)

In [24]:
public_access

,source,name,latitude,longitude,detail
0,rict,20 Mile,-16.085750,136.548390,WP
1,rict,Adjamarragu,-11.152770,132.474810,WP
2,rict,Akwalirrumanja,-13.863600,136.778640,WP
3,rict,Alamirra,-11.067680,132.528170,WP
4,rict,Alice Well,-24.881310,134.229640,WP
...,...,...,...,...,...
379,stand,Alyangula Health Clinic,-13.848046,136.421216,Yes
380,stand,Alyangula Police Station,-13.852252,136.421650,Yes
381,stand,Port Keats (Wadeye) Health Clinic,-14.241205,129.520241,Yes
382,stand,Port Keats (Wadeye) Police Station,-14.239996,129.520903,Yes


In [26]:
out_csv = "data/new_processed/public_access.csv"
public_access.to_csv(out_csv, index=False)
gpd.GeoDataFrame(
    public_access.copy(),
    geometry=gpd.points_from_xy(public_access["longitude"], public_access["latitude"]),
    crs=CRS_LATLON,
).to_file("data/new_processed/public_access.geojson", driver="GeoJSON")

